In [1]:
!pip install datasets transformers torch

from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaForQuestionAnswering,
    DistilBertTokenizerFast,
    DistilBertForQuestionAnswering,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)
import torch

# Define the filter_examples function
def filter_examples(example):
    # Filter out examples where the question or context is empty
    return len(example['question'].strip()) > 0 and len(example['context'].strip()) > 0

# Load the dataset
print("Loading the dataset...")
ds = load_dataset("rajpurkar/squad_v2")

# Load the tokenizer and model (using DistilBERT for faster training)
print("Loading tokenizer and model...")
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForQuestionAnswering.from_pretrained("distilbert-base-uncased")

# Define the preprocessing function
def preprocess_data(examples):
    # Tokenize the context and question with padding and truncation
    inputs = tokenizer(
        examples['question'],
        examples['context'],
        padding='max_length',
        truncation=True,
        max_length=384,  # Reduced max sequence length for speed
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answers = examples["answers"][i]

        if len(answers["answer_start"]) == 0:  # No answer
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = None
            token_end_index = None

            for idx, (start, end) in enumerate(offsets):
                if start <= start_char < end:
                    token_start_index = idx
                if start < end_char <= end:
                    token_end_index = idx
                    break

            start_positions.append(token_start_index if token_start_index is not None else 0)
            end_positions.append(token_end_index if token_end_index is not None else 0)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Apply filtering and preprocessing
print("Filtering and preprocessing the dataset...")
filtered_ds = ds.filter(filter_examples)

# Use smaller subsets of the dataset for faster training
print("Using a smaller subset of the dataset...")
small_train = filtered_ds["train"].shuffle(seed=42).select(range(1000))  # 1000 samples for training
small_eval = filtered_ds["validation"].shuffle(seed=42).select(range(200))  # 200 samples for validation

# Tokenize the datasets
tokenized_train = small_train.map(preprocess_data, batched=True, remove_columns=small_train.column_names)
tokenized_eval = small_eval.map(preprocess_data, batched=True, remove_columns=small_eval.column_names)

# Define data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set training arguments
print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # Larger batch size
    per_device_eval_batch_size=16,
    num_train_epochs=1,  # Fewer epochs for faster training
    weight_decay=0.01,
    logging_dir="./logs",
    fp16=True,  # Mixed precision for speed
    save_strategy="epoch"  # Save checkpoint after each epoch
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize Trainer
print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

# Start training
print("Training the model...")
trainer.train()

# Save the fine-tuned model
final_model_path = "./final_model"
print(f"Saving the fine-tuned model to: {final_model_path}")
tokenizer.save_pretrained(final_model_path)
model.save_pretrained(final_model_path)

# Reload the saved model for inference
print("Reloading the saved model for verification...")
tokenizer = DistilBertTokenizerFast.from_pretrained(final_model_path)
model = DistilBertForQuestionAnswering.from_pretrained(final_model_path)

# Create a QA pipeline
qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Example usage
print("Running inference...")
question = "What is the capital of France?"
context = "France is a country in Europe. The capital of France is Paris."

result = qa_pipeline(question=question, context=context)
print(f"Answer: {result['answer']}, Score: {result['score']}")


Loading the dataset...


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Initializing Trainer...
Training the model...


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: fl506433 (fl506433-wayamba-university-of-sri-lanka). Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,No log,3.853189


Saving the fine-tuned model to: ./final_model
Reloading the saved model for verification...
Running inference...
Answer: Paris, Score: 0.008771009743213654


In [3]:
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForQuestionAnswering,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)
import torch

# Define the filter_examples function
def filter_examples(example):
    # Filter out examples where the question or context is empty
    return len(example['question'].strip()) > 0 and len(example['context'].strip()) > 0

# Load the dataset
print("Loading the dataset...")
ds = load_dataset("rajpurkar/squad_v2")

# Load the tokenizer and model (using DistilBERT for faster training)
print("Loading tokenizer and model...")
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForQuestionAnswering.from_pretrained("distilbert-base-uncased")

# Define the preprocessing function
def preprocess_data(examples):
    # Tokenize the context and question with padding and truncation
    inputs = tokenizer(
        examples['question'],
        examples['context'],
        padding='max_length',
        truncation=True,
        max_length=384,  # Reduced max sequence length for speed
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answers = examples["answers"][i]

        if len(answers["answer_start"]) == 0:  # No answer
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = None
            token_end_index = None

            for idx, (start, end) in enumerate(offsets):
                if start <= start_char < end:
                    token_start_index = idx
                if start < end_char <= end:
                    token_end_index = idx
                    break

            start_positions.append(token_start_index if token_start_index is not None else 0)
            end_positions.append(token_end_index if token_end_index is not None else 0)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Apply filtering and preprocessing
print("Filtering and preprocessing the dataset...")
filtered_ds = ds.filter(filter_examples)

# Use smaller subsets of the dataset for faster training
print("Using a smaller subset of the dataset...")
small_train = filtered_ds["train"].shuffle(seed=42).select(range(1000))  # 1000 samples for training
small_eval = filtered_ds["validation"].shuffle(seed=42).select(range(200))  # 200 samples for validation

# Tokenize the datasets
tokenized_train = small_train.map(preprocess_data, batched=True, remove_columns=small_train.column_names)
tokenized_eval = small_eval.map(preprocess_data, batched=True, remove_columns=small_eval.column_names)

# Define data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set training arguments
print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # Larger batch size
    per_device_eval_batch_size=16,
    num_train_epochs=1,  # Fewer epochs for faster training
    weight_decay=0.01,
    logging_dir="./logs",
    fp16=True,  # Mixed precision for speed
    save_strategy="epoch"  # Save checkpoint after each epoch
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize Trainer
print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

# Start training
print("Training the model...")
trainer.train()

# Save the fine-tuned model
final_model_path = "./final_model"
print(f"Saving the fine-tuned model to: {final_model_path}")
tokenizer.save_pretrained(final_model_path)
model.save_pretrained(final_model_path)

# Reload the saved model for inference
print("Reloading the saved model for verification...")
tokenizer = DistilBertTokenizerFast.from_pretrained(final_model_path)
model = DistilBertForQuestionAnswering.from_pretrained(final_model_path)

# Create a QA pipeline
qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Example usage
print("Running inference...")
question = "What is the capital of France?"
context = "France is a country in Europe. The capital of France is Paris."

result = qa_pipeline(question=question, context=context)
print(f"Answer: {result['answer']}, Score: {result['score']}")


Loading the dataset...
Loading tokenizer and model...


Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Filtering and preprocessing the dataset...
Using a smaller subset of the dataset...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Setting up training arguments...
Initializing Trainer...
Training the model...


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,3.698110


Saving the fine-tuned model to: ./final_model
Reloading the saved model for verification...
Running inference...
Answer: France is a country in Europe. The capital of France is Paris, Score: 0.00865873321890831


In [4]:
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)
import torch

# Define the filter_examples function
def filter_examples(example):
    # Filter out examples where the question or context is empty
    return len(example['question'].strip()) > 0 and len(example['context'].strip()) > 0

# Load the dataset
print("Loading the dataset...")
ds = load_dataset("rajpurkar/squad_v2")

# Load the tokenizer and model (using BERT for question answering)
print("Loading tokenizer and model...")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")

# Define the preprocessing function
def preprocess_data(examples):
    # Tokenize the context and question with padding and truncation
    inputs = tokenizer(
        examples['question'],
        examples['context'],
        padding='max_length',
        truncation=True,
        max_length=384,  # Reduced max sequence length for speed
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answers = examples["answers"][i]

        if len(answers["answer_start"]) == 0:  # No answer
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = None
            token_end_index = None

            for idx, (start, end) in enumerate(offsets):
                if start <= start_char < end:
                    token_start_index = idx
                if start < end_char <= end:
                    token_end_index = idx
                    break

            start_positions.append(token_start_index if token_start_index is not None else 0)
            end_positions.append(token_end_index if token_end_index is not None else 0)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Apply filtering and preprocessing
print("Filtering and preprocessing the dataset...")
filtered_ds = ds.filter(filter_examples)

# Use smaller subsets of the dataset for faster training
print("Using a smaller subset of the dataset...")
small_train = filtered_ds["train"].shuffle(seed=42).select(range(1000))  # 1000 samples for training
small_eval = filtered_ds["validation"].shuffle(seed=42).select(range(200))  # 200 samples for validation

# Tokenize the datasets
tokenized_train = small_train.map(preprocess_data, batched=True, remove_columns=small_train.column_names)
tokenized_eval = small_eval.map(preprocess_data, batched=True, remove_columns=small_eval.column_names)

# Define data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set training arguments
print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=3e-5,  # BERT often benefits from slightly higher learning rates
    per_device_train_batch_size=8,  # Use smaller batch sizes for BERT (heavier model)
    per_device_eval_batch_size=8,
    num_train_epochs=1,  # Fewer epochs for faster training
    weight_decay=0.01,
    logging_dir="./logs",
    fp16=True,  # Mixed precision for speed
    save_strategy="epoch"  # Save checkpoint after each epoch
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize Trainer
print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

# Start training
print("Training the model...")
trainer.train()

# Save the fine-tuned model
final_model_path = "./final_model"
print(f"Saving the fine-tuned model to: {final_model_path}")
tokenizer.save_pretrained(final_model_path)
model.save_pretrained(final_model_path)

# Reload the saved model for inference
print("Reloading the saved model for verification...")
tokenizer = BertTokenizerFast.from_pretrained(final_model_path)
model = BertForQuestionAnswering.from_pretrained(final_model_path)

# Create a QA pipeline
qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Example usage
print("Running inference...")
question = "What is the capital of France?"
context = "France is a country in Europe. The capital of France is Paris."

result = qa_pipeline(question=question, context=context)
print(f"Answer: {result['answer']}, Score: {result['score']}")


Loading the dataset...
Loading tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Setting up training arguments...
Initializing Trainer...
Training the model...


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,2.766431


Saving the fine-tuned model to: ./final_model
Reloading the saved model for verification...
Running inference...
Answer: France is a country in Europe. The capital of France is Paris, Score: 0.002423941856250167
